In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

#import personnal tools
import sys
sys.path.append('../tools/')
from info import *
from imports import *
from tools_generic import *
from events import *

# Load files

In [ ]:
site_list=["d17","d47","d85","dmc"]
file_start_date = '20241201'
file_end_date = '20260630'

golden_start_date="2025-01-01"
golden_end_date="2025-03-31"

In [ ]:
data = {}
daily_data = {}

In [ ]:
data = create_data(site_list, sensors, file_start_date, file_end_date)

# Add the early WIND observations stored in separate WIND_BEGINNING files.
# WARNING: Existing *_wind values take priority; beginning-file values only fill gaps.
data = add_wind_beginning_data(
    data,
    sites=["d17", "d47", "d85", "dmc"],
    data_folder_path="../../data",
    start_date="20241201",
    end_date="20250313",
)

data

In [ ]:
# Extract period of interest (golden month)
data = filter_datasets_golden(
    data, start_date=golden_start_date, end_date=golden_end_date
 )

In [ ]:
# create daily
daily_data = create_daily_data(data)

# Stats

In [ ]:
variable = 'FluxMean1'
compute_variable_stats(data, variable)

In [ ]:
var = "FluxMean1_flowcapt"
plot_binned_distribution(data, var, bin_number=30, min_value=0, max_value=200)


In [ ]:
var = "snowflux_spc"
plot_binned_distribution(data, var, bin_number=30, min_value=0, max_value=200)


# Basic plotting

## Time plots

In [ ]:
variables = ["wdir_wind", "wspd1_wind"]

plot_per_var_multiple_sites(
    sensor_datasets=data,
    variables=variables,
    sites=["d17", "d47", "d85"],
    ymin=[0, 0],
    ymax=[360, 26],
)


In [ ]:
variables = ["FluxMean1_flowcapt", "FluxMean2_flowcapt", "snowflux_spc"]

plot_per_site_multiple_vars(
    data,
    variables,
    sites=["d17", "d47", "d85"],
    figsize=(15, 5),
    ymax=30,
)


## Scatters

In [ ]:
var1 = "wdir_wind"
var2 = "wspd1_wind"
site1 = "d17"
site2 = "d17"
plot_bivariate_scatter(
    data,
    var1=var1,
    var2=var2,
    site1=site1,
    site2=site2,
    show_corr=False,
    show_fit=False,
    min_val=[0, 0],
    max_val=[360, 26],
    figsize=(7, 6),
)


In [ ]:
var1 = "FluxMean2_flowcapt"
var2 = "snowflux_spc"
var3 = "wspd1_wind"
site1 = "d47"
site2 = "d47"
site3 = "d47"
plot_trivariate_scatter(
    data,
    var1=var1,
    var2=var2,
    var3=var3,
    site1=site1,
    site2=site2,
    site3=site3,
    min3=10,
    max3=20,
    max_val=[300, 300],
    figsize=(7, 6),
    show_oneone=True,
)


# Events

In [ ]:
detector = EventDetector(
    threshold=1.0,
    min_timesteps=12,
    buffer_timesteps=0,
)

sampled_data = create_resampled_data(data, "30min")
additional_variables = [
    "FluxMean1_flowcapt",
    "snowflux_spc",
    "wspd1_wind",
    "wspd2_wind",
    "wdir_wind",
    "Hagl_flowcapt",
    "T1_surf",
    "RH1_surf",
]
collection = detector.detect_events(
    sampled_data,
    variable="FluxMean2_flowcapt",
    additional_variables=additional_variables,
)
collection_d17 = collection.get_site("d17")
collection_d47 = collection.get_site("d47")
non_collection = detector.detect_non_events(
    sampled_data,
    variable="FluxMean2_flowcapt",
    additional_variables=additional_variables,
)
non_collection_d17 = non_collection.get_site("d17")
non_collection_d47 = non_collection.get_site("d47")


In [ ]:
collection_d17.to_catalog(variables=["FluxMean2"])#.head()

In [ ]:
non_collection_d17.to_catalog(variables=["FluxMean2"])#.head()

In [ ]:
collection_d47.to_catalog(variables=["FluxMean2"])

In [ ]:
# plot_single_event(collection[2], 
#                     ['FluxMean1','FluxMean2','snowflux','wspd1'])

In [ ]:
print_duration_stats(collection)
print_duration_stats(collection_d17)
print_duration_stats(collection_d47)

In [ ]:
print_integrated_flux_stats(collection)
print_integrated_flux_stats(collection_d17)
print_integrated_flux_stats(collection_d47)

In [ ]:
plot_diurnal_start_distribution(collection_d17)

### Time series

In [ ]:
plot_event_collection_traces(
    collection=collection_d17,
    variable="wspd1_wind",
    align_to="start_time",
    time_unit="h",
    cmap_name="viridis",
    alpha=1,
    linewidth=0.8,
    show_mean=False,
)


In [ ]:
plot_event_collection_traces(
    collection=non_collection_d17,
    variable="FluxMean2_flowcapt",
    align_to="start_time",
    time_unit="h",
    cmap_name="viridis",
    alpha=1,
    linewidth=0.7,
)


In [ ]:
plot_event_collection_traces(
    collection=collection_d47,
    variable="wspd1_wind",
    align_to="start_time",
    time_unit="h",
    cmap_name="viridis",
    alpha=1,
    linewidth=0.7,
    show_mean=False,
)


In [ ]:
varlist = ["FluxMean2_flowcapt", "wspd1_wind", "wdir_wind"]
plot_events_vs_nonevents_chronological(
    collection_d17,
    non_collection_d17,
    variables=varlist,
    figsize=(15, 15),
)
plot_events_vs_nonevents_chronological(
    collection_d47,
    non_collection_d47,
    variables=varlist,
    figsize=(15, 15),
)


In [ ]:
varlist = ['FluxMean2','FluxMean1','snowflux','wspd1']
varlist = ['FluxMean2','Hagl','T1','RH1']
# plot_events_vs_nonevents_chronological(
#     collection_d47,
#     non_collection_d47,
#     variables=varlist
# )

In [ ]:
var1 = "wspd1_wind"
var2 = "FluxMean2_flowcapt"
plot_collection_bivariate_scatter(
    collection=collection,
    var1=var1,
    var2=var2,
    show_corr=False,
    show_fit=False,
)
plot_collection_bivariate_scatter(
    collection=non_collection,
    var1=var1,
    var2=var2,
    show_corr=False,
    show_fit=False,
)


### Composites

In [ ]:
composite = compute_event_composite(
    events=collection,
    variables=["FluxMean1_flowcapt", "FluxMean2_flowcapt", "snowflux_spc"],
    align_to="start_time",
    time_unit="h",
)
composite_d17 = compute_event_composite(
    events=collection_d17,
    variables=["FluxMean1_flowcapt", "FluxMean2_flowcapt", "snowflux_spc"],
    align_to="start_time",
    time_unit="h",
)
composite_d47 = compute_event_composite(
    events=collection_d47,
    variables=["FluxMean1_flowcapt", "FluxMean2_flowcapt", "snowflux_spc"],
    align_to="start_time",
    time_unit="h",
)


In [ ]:
plot_event_composite(
    composite_ds=composite,
    variables=["FluxMean1_flowcapt", "FluxMean2_flowcapt", "snowflux_spc", "wspd1_wind"],
    use_quantiles=True,
)


In [ ]:
plot_event_composite(
    composite_ds=composite_d17,
    variables=["FluxMean1_flowcapt", "FluxMean2_flowcapt", "snowflux_spc", "wspd1_wind"],
    use_quantiles=True,
)


In [ ]:
plot_event_composite(
    composite_ds=composite_d47,
    variables=["FluxMean1_flowcapt", "FluxMean2_flowcapt", "snowflux_spc", "wspd1_wind"],
    use_quantiles=True,
)


# MRR

In [ ]:
mrr_site='d17'
zea_dbz_10mn = xr.open_dataset(f'../../data/MRR_aggregated/zea_averaged10mn_mrr_{mrr_site}_20250201_20250228.nc')
zea_dbz_10mn

In [ ]:
zea_dbz_30mn = resample_dbz(zea_dbz_10mn, 
                            'Zea',
                            '30min')

In [ ]:
print(zea_dbz_30mn)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
format_time_plot(zea_dbz_10mn['Zea'].mean(dim='range'),ax)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
format_time_plot(zea_dbz_10mn['Zea'].mean(dim='time'),ax)

In [ ]:
# The MRR variable is stored in data, not retroactively injected into old events.
# To include it in events, rerun event detection later with it in additional_variables.


In [ ]:
precip_fraction = compute_vertical_over_threshold_fraction(
    zea_dbz_30mn,
    "Zea",
    "range",
    1.0,
    (0, 1000),
).rename("precip_fraction_mrr")

data = add_variable_to_data(
    data,
    site=mrr_site,
    data_array=precip_fraction,
    variable_name="precip_fraction_mrr",
)

precip_fraction.plot()


In [ ]:
data[mrr_site]["precip_fraction_mrr"]


In [ ]:
# Optional alternative height interval. Keep a distinct name if both products are needed.
precip_fraction_500_1500 = compute_vertical_over_threshold_fraction(
    zea_dbz_30mn,
    "Zea",
    "range",
    1.0,
    (500, 1500),
).rename("precip_fraction_mrr_500_1500")

# Uncomment to store this alternative product as well:
# data = add_variable_to_data(
#     data,
#     site=mrr_site,
#     data_array=precip_fraction_500_1500,
#     variable_name="precip_fraction_mrr_500_1500",
# )
precip_fraction_500_1500.plot()


In [ ]:
varlist = ["FluxMean2_flowcapt", "FluxMean1_flowcapt", "mrr_zea_fraction_above1dBZ"]
plot_events_vs_nonevents_chronological(
    collection_d17,
    non_collection_d17,
    variables=varlist,
    figsize=(12, 6),
)


In [ ]:
threshold_func = partial(classify_by_mean_threshold, variable="mrr_zea_fraction_above1dBZ", threshold=0.1)
precip_clusters = collection_d17.classify(threshold_func)

In [ ]:
precip_clusters

In [ ]:
plot_event_collection_traces(precip_clusters['Low_Intensity'],'mrr_zea_fraction_above1dBZ',align_to='start_time', alpha=0.8, linewidth=2)

In [ ]:
plot_event_collection_traces(precip_clusters['High_Intensity'],'mrr_zea_fraction_above1dBZ',align_to='start_time', alpha=0.8, linewidth=2)

In [ ]:
# Optional: rerun event detection after adding the MRR variable.
# Include "precip_fraction_mrr" only when it should be part of each event dataset.
# mrr_detector = EventDetector(threshold=1.0, min_timesteps=12, buffer_timesteps=0)
# mrr_collection = mrr_detector.detect_events(
#     create_resampled_data(data, "30min"),
#     variable="FluxMean2_flowcapt",
#     additional_variables=["wspd1_merged", "precip_fraction_mrr"],
# )
